In [11]:
"""
BioLogic .mpr batch analysis
Only use I, E, t for plotting
================================

推荐命名格式
------------
倍率测试（GCPL / rate）:
    1c.mpr
    2c.mpr
    5c.mpr
    10c.mpr
    20c.mpr
    1c-f.mpr

CV:
    cv.mpr

功能
----
1. 批量读取文件夹内所有 .mpr
2. 用户输入:
   - 数据文件夹路径
   - 活性物质质量 mass (mg)
3. 对 GCPL / rate:
   - 只用 time, voltage, current
   - 不使用 cycle number / half cycle / Q
   - 自动重建 half-cycle
   - 自动取第2个完整 cycle
   - 计算 specific capacity (mAh/g)
   - 画 Voltage - Specific Capacity
4. 对 CV:
   - 只用 E 和 I
   - 画 Current - Voltage
5. 导出 CSV、图片和 summary

说明
----
对于 GCPL：
- 使用平滑后的电流符号来分段
- 每个连续同号段视为一个 half-cycle
- 对每个 half-cycle 用真实电流和真实 dt 积分容量
- 每两个相邻 half-cycle 视为一个完整 cycle
- 默认取第 2 个完整 cycle 作图

重要
----
这版故意只依赖最原始的:
- t
- E
- I

不依赖 BioLogic 附带的:
- cycle number
- half cycle
- Ns
- Q 列
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from galvani import BioLogic


# =========================
# USER CONFIG
# =========================
EXPORT_CSV = True
DPI = 300
FIGSIZE = (6.2, 4.6)
SHOW_DEBUG = True

# 平滑和分段参数
ROLLING_WINDOW = 11          # 电流平滑窗口
MIN_SEGMENT_POINTS = 20      # 少于这个点数的短段会被并入前后段
# =========================


# ---------- 输入 ----------
def ask_data_dir():
    while True:
        path_str = input("请输入 .mpr 文件夹路径（例如 ./data 或 C:/xxx/xxx）: ").strip()
        if path_str == "":
            path = Path("./data")
            print("使用默认路径: ./data")
        else:
            path = Path(path_str)

        if path.exists() and path.is_dir():
            return path
        print("路径不存在或不是文件夹，请重新输入。")


def ask_mass_mg():
    while True:
        try:
            mass_mg = float(input("请输入活性物质质量 mass（mg）: "))
            if mass_mg <= 0:
                print("mass 必须大于 0，请重新输入。")
                continue
            return mass_mg
        except ValueError:
            print("输入无效，请输入数字。")


# ---------- 文件识别 ----------
def parse_filename(file_stem: str):
    name = file_stem.lower().strip()

    if "cv" in name:
        return {"kind": "cv", "label": "cv"}

    m = re.search(r"(^|[_-])(\d+(?:\.\d+)?)c(-f)?($|[_-])", name)
    if m:
        rate = m.group(2) + "c"
        suffix = "-f" if m.group(3) else ""
        return {"kind": "rate", "label": rate + suffix}

    if "gcpl" in name:
        return {"kind": "rate", "label": "gcpl"}

    return {"kind": "unknown", "label": name}


# ---------- 读取 mpr ----------
def read_mpr(file_path: Path):
    mpr = BioLogic.MPRfile(str(file_path))
    return pd.DataFrame(mpr.data)


# ---------- 找列 ----------
def normalize_col_name(col):
    return str(col).strip().lower()


def find_first_matching_column(df, priority_patterns):
    cols = list(df.columns)
    cols_lower = [normalize_col_name(c) for c in cols]

    for pat in priority_patterns:
        for col, low in zip(cols, cols_lower):
            if pat in low:
                return col
    return None


def get_time_column(df):
    return find_first_matching_column(df, [
        "time/s",
        "time (s)",
        "test time",
        "time"
    ])


def get_voltage_column(df):
    return find_first_matching_column(df, [
        "ewe/v",
        "voltage",
        "ecell",
        "potential"
    ])


def get_current_column(df):
    return find_first_matching_column(df, [
        "<i>/ma",
        "i/ma",
        "current",
        "control/v/ma",
        "control/ma"
    ])


# ---------- 数值工具 ----------
def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def clean_raw_t_e_i(df):
    t_col = get_time_column(df)
    e_col = get_voltage_column(df)
    i_col = get_current_column(df)

    if t_col is None:
        raise ValueError("没找到时间列")
    if e_col is None:
        raise ValueError("没找到电压列")
    if i_col is None:
        raise ValueError("没找到电流列")

    out = pd.DataFrame({
        "t": to_num(df[t_col]),
        "E": to_num(df[e_col]),
        "I": to_num(df[i_col]),
    })

    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    out = out.sort_values("t").reset_index(drop=True)

    return out, t_col, e_col, i_col


def compute_dt_seconds(t_series):
    t = to_num(t_series).to_numpy()
    dt = np.diff(t, prepend=t[0])
    dt[dt < 0] = np.nan
    if len(dt) > 1:
        valid = np.isfinite(dt[1:]) & (dt[1:] > 0)
        median_dt = np.median(dt[1:][valid]) if np.any(valid) else 0.0
        if not np.isfinite(dt[0]) or dt[0] <= 0:
            dt[0] = median_dt
    dt = np.nan_to_num(dt, nan=0.0)
    return dt


# ---------- 电流符号重建 ----------
def smooth_current_sign(I_series, rolling_window=11):
    """
    对电流轻微平滑后取符号。
    不使用固定绝对阈值，适合小电流。
    """
    I = pd.Series(I_series).astype(float)
    I_smooth = I.rolling(rolling_window, center=True, min_periods=1).median()

    sign = np.sign(I_smooth.to_numpy())

    # 把 0 用邻居符号填掉
    sign = sign.astype(float)

    # 前向填充
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i - 1]

    # 后向补开头
    if len(sign) > 1 and sign[0] == 0:
        for i in range(1, len(sign)):
            if sign[i] != 0:
                sign[0] = sign[i]
                break

    # 仍为0则保持0
    return I_smooth.to_numpy(), sign


def initial_segments_from_sign(sign_array):
    """
    根据符号序列切连续段
    返回 [(start, end, sign), ...]，end 不包含
    """
    if len(sign_array) == 0:
        return []

    segments = []
    start = 0
    current_sign = sign_array[0]

    for i in range(1, len(sign_array)):
        if sign_array[i] != current_sign:
            segments.append((start, i, current_sign))
            start = i
            current_sign = sign_array[i]

    segments.append((start, len(sign_array), current_sign))
    return segments


def merge_short_segments(segments, min_points=20):
    """
    把过短的符号段并入相邻段，减少单点翻转造成的伪 half-cycle
    """
    if len(segments) <= 1:
        return segments

    segs = segments[:]
    changed = True

    while changed and len(segs) > 1:
        changed = False
        new_segs = []
        i = 0

        while i < len(segs):
            s, e, sg = segs[i]
            length = e - s

            if length >= min_points:
                new_segs.append((s, e, sg))
                i += 1
                continue

            changed = True

            # 首段短：并到后段
            if i == 0 and len(segs) > 1:
                s2, e2, sg2 = segs[i + 1]
                new_segs.append((s, e2, sg2))
                i += 2
                continue

            # 末段短：并到前段
            if i == len(segs) - 1 and len(new_segs) > 0:
                s_prev, e_prev, sg_prev = new_segs[-1]
                new_segs[-1] = (s_prev, e, sg_prev)
                i += 1
                continue

            # 中间短：优先并到前段
            if len(new_segs) > 0:
                s_prev, e_prev, sg_prev = new_segs[-1]
                new_segs[-1] = (s_prev, e, sg_prev)
                i += 1
            else:
                new_segs.append((s, e, sg))
                i += 1

        # 合并后如果相邻符号一样，再拼起来
        compact = []
        for seg in new_segs:
            if not compact:
                compact.append(seg)
            else:
                s0, e0, sg0 = compact[-1]
                s1, e1, sg1 = seg
                if sg0 == sg1:
                    compact[-1] = (s0, e1, sg0)
                else:
                    compact.append(seg)

        segs = compact

    return segs


def rebuild_half_cycles(df_eit):
    """
    输入含 t, E, I 的 dataframe
    
    输出 half-cycle 列表
    每个 half-cycle:
        {
            'sign': +1/-1,
            'name': 'Charge' / 'Discharge',
            'data': dataframe with t,E,I,Q_mAh,q_mAh_g
        }
    """
    df = df_eit.copy().reset_index(drop=True)

    I_smooth, sign = smooth_current_sign(df["I"], rolling_window=ROLLING_WINDOW)
    df["I_smooth"] = I_smooth
    df["sign"] = sign

    segs = initial_segments_from_sign(df["sign"].to_numpy())
    segs = [seg for seg in segs if seg[2] != 0]
    segs = merge_short_segments(segs, min_points=MIN_SEGMENT_POINTS)

    if SHOW_DEBUG:
        print("重建出的原始 half-cycle 段数 =", len(segs))
        print("segments =", [(s, e, int(sg), e - s) for s, e, sg in segs])

    half_cycles = []

    for idx, (s, e, sg) in enumerate(segs):
        sub = df.iloc[s:e].copy().reset_index(drop=True)
        if len(sub) < 2:
            continue

        dt = compute_dt_seconds(sub["t"])
        Q_mAh = np.cumsum(np.abs(sub["I"].to_numpy()) * dt) / 3600.0
        sub["Q_mAh"] = Q_mAh

        name = "Charge" if sg > 0 else "Discharge"

        half_cycles.append({
            "index": idx,
            "sign": int(np.sign(sg)),
            "name": name,
            "data": sub
        })

    return half_cycles


def choose_second_full_cycle(half_cycles):
    """
    每两个相邻 half-cycle 视为一个完整 cycle。
    默认取第2个完整 cycle，即 half_cycles[2] 和 half_cycles[3]
    若不足，则退化处理。
    """
    n = len(half_cycles)

    if n >= 4:
        return [half_cycles[2], half_cycles[3]], "reconstructed cycle #2"
    elif n >= 2:
        return [half_cycles[0], half_cycles[1]], "fallback: first full cycle"
    elif n == 1:
        return [half_cycles[0]], "fallback: only one half-cycle"
    else:
        return [], "no usable half-cycle"


# ---------- 作图 ----------
def plot_rate_file_only_EIt(df, label, mass_mg, out_png):
    df_eit, t_col, e_col, i_col = clean_raw_t_e_i(df)

    half_cycles = rebuild_half_cycles(df_eit)
    selected, note = choose_second_full_cycle(half_cycles)

    if len(selected) == 0:
        raise ValueError("无法从 I,E,t 重建有效的 cycle")

    mass_g = mass_mg / 1000.0

    plt.figure(figsize=FIGSIZE)

    plotted_any = False
    branch_names = []

    for hc in selected:
        sub = hc["data"].copy()
        sub["q_mAh_g"] = sub["Q_mAh"] / mass_g

        x = to_num(sub["q_mAh_g"]).to_numpy()
        y = to_num(sub["E"]).to_numpy()
        mask = np.isfinite(x) & np.isfinite(y)

        if np.any(mask):
            plt.plot(x[mask], y[mask], linewidth=1.2, label=hc["name"])
            plotted_any = True
            branch_names.append(hc["name"])

    if not plotted_any:
        plt.close()
        raise ValueError("没有有效的 GCPL 数据可画")

    plt.xlabel("Specific capacity (mAh/g)")
    plt.ylabel("Voltage (V vs. Li/Li )")
    plt.title(f"Rate test: {label} ({note})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    return {
        "note": note,
        "time_col": t_col,
        "voltage_col": e_col,
        "current_col": i_col,
        "n_half_cycles_rebuilt": len(half_cycles),
        "branches_plotted": "; ".join(branch_names),
    }


def plot_cv_file_only_EIt(df, label, out_png):
    df_eit, t_col, e_col, i_col = clean_raw_t_e_i(df)

    x = df_eit["E"].to_numpy()
    y = df_eit["I"].to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)

    if not np.any(mask):
        raise ValueError("CV 没有有效数据可画")

    plt.figure(figsize=FIGSIZE)
    plt.plot(x[mask], y[mask], linewidth=1.0)
    plt.xlabel("Voltage (V)")
    plt.ylabel("Current (mA)")
    plt.title(f"CV: {label}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

    return {
        "time_col": t_col,
        "voltage_col": e_col,
        "current_col": i_col,
        "n_points": int(np.sum(mask)),
    }


# ---------- 主程序 ----------
def main():
    data_dir = ask_data_dir()
    mass_mg = ask_mass_mg()

    export_dir = data_dir / "output"
    csv_dir = export_dir / "csv"
    fig_dir = export_dir / "figures"

    export_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(data_dir.glob("*.mpr"))
    if not files:
        print(f"在 {data_dir} 中没有找到 .mpr 文件。")
        return

    summary = []

    print(f"\n共找到 {len(files)} 个 .mpr 文件")
    print(f"mass = {mass_mg} mg\n")

    for fp in files:
        meta = parse_filename(fp.stem)

        print("=" * 90)
        print(f"读取文件: {fp.name}")
        print(f"识别类型: {meta['kind']}")
        print(f"标签: {meta['label']}")

        try:
            df = read_mpr(fp)

            print(f"行数: {len(df)}")
            print("列名:")
            print(list(df.columns))

            # 原始 csv 导出
            csv_path = None
            if EXPORT_CSV:
                csv_path = csv_dir / f"{fp.stem}.csv"
                df.to_csv(csv_path, index=False, encoding="utf-8-sig")
                print(f"已导出 CSV: {csv_path.name}")

            fig_path = None
            plot_info = {}

            if meta["kind"] == "rate":
                fig_path = fig_dir / f"{fp.stem}_rate.png"
                plot_info = plot_rate_file_only_EIt(df, meta["label"], mass_mg, fig_path)
                print(f"已导出 rate 图: {fig_path.name}")

            elif meta["kind"] == "cv":
                fig_path = fig_dir / f"{fp.stem}_cv.png"
                plot_info = plot_cv_file_only_EIt(df, meta["label"], fig_path)
                print(f"已导出 CV 图: {fig_path.name}")

            else:
                print("未识别类型，跳过画图。")

            summary.append({
                "file": fp.name,
                "kind": meta["kind"],
                "label": meta["label"],
                "n_rows": len(df),
                "csv_exported": str(csv_path) if csv_path else "",
                "figure_exported": str(fig_path) if fig_path else "",
                **plot_info
            })

        except Exception as e:
            print(f"处理失败: {e}")
            summary.append({
                "file": fp.name,
                "kind": meta["kind"],
                "label": meta["label"],
                "n_rows": None,
                "error": str(e),
            })

    summary_df = pd.DataFrame(summary)
    summary_path = export_dir / "summary.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 90)
    print("全部处理完成")
    print(f"结果文件夹: {export_dir}")
    print(f"汇总表: {summary_path}")


if __name__ == "__main__":
    main()

请输入 .mpr 文件夹路径（例如 ./data 或 C:/xxx/xxx）:  /Users/leili/Downloads/quinn
请输入活性物质质量 mass（mg）:  1.3



共找到 1 个 .mpr 文件
mass = 1.3 mg

读取文件: QN047_01_42_01_CVA_C03.mpr
识别类型: cv
标签: cv
行数: 25031
列名:
['flags', 'time/s', 'control/V', 'Ewe/V', '<I>/mA', 'cycle number', '(Q-Qo)/C', 'I Range', 'Ns']
已导出 CSV: QN047_01_42_01_CVA_C03.csv
已导出 CV 图: QN047_01_42_01_CVA_C03_cv.png

全部处理完成
结果文件夹: /Users/leili/Downloads/quinn/output
汇总表: /Users/leili/Downloads/quinn/output/summary.csv
